In [6]:
import sys
import os
import subprocess
import platform
from pathlib import Path


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def ejecutar_comando(comando):
    """Ejecuta un comando del sistema y devuelve su resultado."""
    try:
        resultado = subprocess.run(
            comando,
            capture_output=True,
            text=True,
            check=False
        )

        if resultado.returncode == 0:
            return resultado.stdout.strip()

        return None

    except FileNotFoundError:
        return None


def encontrar_raiz_git(ruta):
    """
    Busca la raíz del repositorio Git a partir de la ubicación actual.
    La ruta se utiliza internamente y nunca se muestra.
    """
    ruta = Path(ruta).resolve()

    for directorio in [ruta] + list(ruta.parents):

        if (directorio / ".git").exists():
            return directorio

    return None


# ============================================================
# INFORMACIÓN DE PYTHON
# ============================================================

print("=" * 60)
print("        DIAGNÓSTICO DEL ENTORNO PYTHON")
print("=" * 60)

print("\n[PYTHON]")

print(f"Versión de Python : {platform.python_version()}")

print(f"Plataforma        : {platform.system()} "
      f"{platform.release()} "
      f"{platform.machine()}")


# ============================================================
# INFORMACIÓN DEL ENTORNO CONDA
# ============================================================

print("\n[CONDA]")

conda_env = os.environ.get("CONDA_DEFAULT_ENV")

if conda_env:
    print(f"Entorno activo    : {conda_env}")
else:
    print("Entorno activo    : No detectado")


# ============================================================
# INFORMACIÓN DEL KERNEL
# ============================================================

print("\n[KERNEL]")

# No mostramos sys.prefix ni sys.base_prefix porque contienen
# rutas locales del sistema.

python_entorno = os.environ.get("CONDA_PREFIX")

if python_entorno and sys.executable.startswith(python_entorno):
    kernel_correcto = True
else:
    kernel_correcto = False

print(f"Kernel Conda      : {'Correcto' if kernel_correcto else 'Revisar'}")


# ============================================================
# INFORMACIÓN DEL PROYECTO
# ============================================================

print("\n[PROYECTO]")

directorio_actual = Path.cwd()

raiz_git = encontrar_raiz_git(directorio_actual)

if raiz_git:

    print("Repositorio Git   : Sí")

else:

    print("Repositorio Git   : No")


# ============================================================
# INFORMACIÓN DE GIT
# ============================================================

print("\n[GIT]")

git_version = ejecutar_comando(["git", "--version"])

if git_version:

    print(f"Versión Git       : {git_version}")
    git_disponible = True

else:

    print("Versión Git       : No disponible")
    git_disponible = False


if raiz_git and git_disponible:

    # Branch actual
    branch = ejecutar_comando(
        [
            "git",
            "-C",
            str(raiz_git),
            "branch",
            "--show-current"
        ]
    )

    print(f"Branch actual     : {branch or 'No disponible'}")


    # Remote:
    # No mostramos la dirección del repositorio para evitar
    # exponer usuario, organización o URL.

    remote = ejecutar_comando(
        [
            "git",
            "-C",
            str(raiz_git),
            "remote"
        ]
    )

    if remote:
        print("Remote GitHub     : Configurado")
    else:
        print("Remote GitHub     : No configurado")


    # Estado de Git
    status = ejecutar_comando(
        [
            "git",
            "-C",
            str(raiz_git),
            "status",
            "--short"
        ]
    )

    if status:

        # Contamos los cambios sin mostrar nombres de archivos.
        cambios = len(status.splitlines())

        print(f"Estado            : Hay {cambios} cambio(s) pendiente(s)")

    else:

        print("Estado            : Working tree limpio")


# ============================================================
# RESUMEN
# ============================================================

print("\n" + "=" * 60)
print("                     RESUMEN")
print("=" * 60)

print(f"Python             : {platform.python_version()}")
print(f"Entorno Conda      : {conda_env or 'No detectado'}")
print(f"Kernel Conda       : {'Correcto' if kernel_correcto else 'Revisar'}")
print(f"Repositorio Git    : {'Sí' if raiz_git else 'No'}")
print(f"Git disponible     : {'Sí' if git_disponible else 'No'}")

print("=" * 60)

        DIAGNÓSTICO DEL ENTORNO PYTHON

[PYTHON]
Versión de Python : 3.11.15
Plataforma        : Darwin 25.5.0 arm64

[CONDA]
Entorno activo    : renova_prestamo_ml_sd

[KERNEL]
Kernel Conda      : Correcto

[PROYECTO]
Repositorio Git   : Sí

[GIT]
Versión Git       : git version 2.50.1 (Apple Git-155)
Branch actual     : develop
Remote GitHub     : Configurado
Estado            : Working tree limpio

                     RESUMEN
Python             : 3.11.15
Entorno Conda      : renova_prestamo_ml_sd
Kernel Conda       : Correcto
Repositorio Git    : Sí
Git disponible     : Sí
